# Spooky Halloween

In [19]:
import pandas as pd
# Random Forest Model
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [20]:
df = pd.read_csv('zombie_predictor_dataset.csv')
df.head(15)

,age,speed_kmh,hunger_level,has_fever,bite_marks,exposure_hours,eye_color,region_risk,smell_decay,groan_intensity,infected
0,67,5.61,0.82,1,0,2.3,blue,medium,0.09,3.71,0
1,87,9.11,3.26,1,1,9.6,blue,medium,7.23,1.18,1
2,81,8.38,7.94,1,1,48.3,black,low,5.91,1.12,1
3,72,3.71,0.00,1,0,34.9,red,medium,0.00,8.15,1
4,88,8.55,5.27,1,1,13.4,red,medium,0.00,6.61,1
5,17,2.94,8.90,0,0,20.1,green,low,0.76,6.19,1
6,37,8.18,9.11,0,0,51.8,green,medium,0.00,5.01,1
7,27,10.86,4.73,1,1,0.9,red,medium,2.30,7.30,1
8,15,6.90,10.00,0,0,5.5,brown,low,1.61,3.65,1
9,83,6.54,5.33,1,0,16.8,green,low,5.94,4.64,0


## Se hace un modelo de Random Forest

In [21]:
# One hot encoding for eye color
eye_color_dummies = pd.get_dummies(df['eye_color'], prefix='eye_color')
df = pd.concat([df, eye_color_dummies], axis=1)
df.drop('eye_color', axis=1, inplace=True)

# Region risk
region_dummies = pd.get_dummies(df['region_risk'], prefix='region_risk')
df = pd.concat([df, region_dummies], axis=1)
df.drop('region_risk', axis=1, inplace=True)

In [22]:
# Preprocessing
X = df.drop('infected', axis=1)
y = df['infected']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model Training
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

# Evaluation
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

# R cuadrado
r2_score = rf_model.score(X_test, y_test)
print(f'R^2 Score: {r2_score}')


[[227 199]
 [103 471]]
              precision    recall  f1-score   support

           0       0.69      0.53      0.60       426
           1       0.70      0.82      0.76       574

    accuracy                           0.70      1000
   macro avg       0.70      0.68      0.68      1000
weighted avg       0.70      0.70      0.69      1000

R^2 Score: 0.698


Exportar modelo

In [23]:
# Graficar el entrenamiento
import matplotlib.pyplot as plt
import numpy as np
import joblib
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]
print("Feature ranking:")
for f in range(X.shape[1]):
    print(f"{f + 1}. feature {X.columns[indices[f]]} ({importances[indices[f]]})")
#plt.figure()
#plt.title("Feature Importances")
#plt.bar(range(X.shape[1]), importances[indices], align="center")
#plt.xticks(range(X.shape[1]), X.columns[indices], rotation=90)
#plt.show()

Feature ranking:
1. feature exposure_hours (0.17612704377558872)
2. feature speed_kmh (0.1573292332497129)
3. feature groan_intensity (0.1443711726815211)
4. feature hunger_level (0.12747652872956935)
5. feature smell_decay (0.11388116009434897)
6. feature age (0.10037627677153689)
7. feature bite_marks (0.03822697139189489)
8. feature has_fever (0.02786789462054972)
9. feature eye_color_red (0.017620256618047326)
10. feature eye_color_brown (0.016582031023569834)
11. feature region_risk_low (0.01610370644729487)
12. feature region_risk_medium (0.014654457594083915)
13. feature eye_color_green (0.013624372126021897)
14. feature eye_color_blue (0.013508126917339394)
15. feature region_risk_high (0.011630612236864636)
16. feature eye_color_black (0.010620155722055503)


In [24]:
# exportar modelo a un archivo .h5
import joblib
joblib.dump(rf_model, 'zombie_rf_model.h5')

['zombie_rf_model.h5']

## Intentar un modelo Sequential

In [25]:
# Modelo Sequential con Keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Crear el modelo
model = Sequential()
model.add(Dense(64, input_dim=X.shape[1], activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

# Train model
model.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.fit(X_train, y_train, epochs=200, batch_size=10, validation_split=0.2)

Epoch 1/200


d:\GIT\TC3007C\.venv\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


320/320 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6591 - loss: 0.6419 - val_accuracy: 0.7200 - val_loss: 0.5473
Epoch 2/200
320/320 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7069 - loss: 0.5737 - val_accuracy: 0.6963 - val_loss: 0.6029
Epoch 3/200
320/320 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7100 - loss: 0.5642 - val_accuracy: 0.7000 - val_loss: 0.6046
Epoch 4/200
320/320 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7175 - loss: 0.5516 - val_accuracy: 0.7225 - val_loss: 0.5724
Epoch 5/200
320/320 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7100 - loss: 0.5525 - val_accuracy: 0.7150 - val_loss: 0.5285
Epoch 6/200
320/320 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7194 - loss: 0.5456 - val_accuracy: 0.7325 - val_loss: 0.5133
Epoch 7/200
320/320 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7231 - loss: 0.5382 - val_accuracy: 0.7200 - val_loss: 0.5462
Epoch 8/200
320/320 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7234 - loss: 0.5378 - val_accuracy: 0.7250

In [26]:
# Test model
loss, accuracy = model.evaluate(X_test, y_test)
print(f'Test Loss: {loss}, Test Accuracy: {accuracy}')

# R2 score for Sequential model

score = model.evaluate(X_test, y_test, verbose=0)
print('Test accuracy:', score)

# Exportar modelo Sequential
model.save('zombie_sequential_model.h5')

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6660 - loss: 0.7646
Test Loss: 0.7646453976631165, Test Accuracy: 0.6660000085830688


Test accuracy: [0.7646453976631165, 0.6660000085830688]
